# Notebook 2: QLoRA SFT Training + Interactive Chat\n\nFine-tune GPT-OSS-120B on preprocessed sales call transcripts using QLoRA.\nAfter training, provides an interactive chat loop that simulates an outbound sales call.\n\n**Input:** `data/train_examples.jsonl` (from notebook 1)\n**Output:** `checkpoints/final_adapter/` (LoRA weights + tokenizer)

In [ ]:
import torch
import json
from pathlib import Path
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from datasets import Dataset

# ============================================================
# Configuration — update MODEL_DIR before running
# ============================================================
MODEL_DIR = "/path/to/gpt-oss-120b"   # <-- FILL IN: local checkpoint directory
DATA_FILE = Path("data/train_examples.jsonl")
OUTPUT_DIR = Path("checkpoints")

# Training hyperparameters
MAX_SEQ_LEN = 4096
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
BATCH_SIZE = 1              # per-device (small for 120B model)
GRAD_ACCUM_STEPS = 8        # effective batch size = 8
NUM_EPOCHS = 3
WARMUP_RATIO = 0.05

## Load Tokenizer & Register Special Tokens\n\nThe 8 special tokens are registered as `additional_special_tokens` so they are\nnever split by BPE. Each becomes a single token ID.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)

SPECIAL_TOKENS = [
    "<|system|>", "<|/system|>",
    "<|context|>", "<|/context|>",
    "<|conversation|>", "<|/conversation|>",
    "<|agent|>",
    "<|customer|>",
]

num_added = tokenizer.add_special_tokens({
    "additional_special_tokens": SPECIAL_TOKENS,
})
print(f"Added {num_added} special tokens. New vocab size: {len(tokenizer)}")

# Store key token IDs for generation stopping
AGENT_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|agent|>")
CUSTOMER_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|customer|>")
CONV_END_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|/conversation|>")

print(f"<|agent|> = {AGENT_TOKEN_ID}")
print(f"<|customer|> = {CUSTOMER_TOKEN_ID}")
print(f"<|/conversation|> = {CONV_END_TOKEN_ID}")

# Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"Set pad_token = eos_token ({tokenizer.eos_token_id})")

## Load & Tokenize Data with Loss Masks\n\nTokenize `prefix` and `target` separately, then concatenate. This guarantees\nan exact mask boundary: everything before the target gets label `-100` (ignored by loss).

In [ ]:
def load_and_tokenize(data_path: Path) -> Dataset:
    """Load JSONL examples and tokenize with loss masks."""
    examples = []
    with open(data_path, "r", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))

    input_ids_list = []
    labels_list = []
    attention_mask_list = []

    for ex in examples:
        # Tokenize prefix and target separately for exact mask boundary
        prefix_ids = tokenizer.encode(ex["prefix"], add_special_tokens=False)
        target_ids = tokenizer.encode(ex["target"], add_special_tokens=False)
        # Append EOS so model learns when to stop generating
        target_ids = target_ids + [tokenizer.eos_token_id]

        full_ids = prefix_ids + target_ids

        # Truncate from LEFT of prefix if too long (keep target intact)
        if len(full_ids) > MAX_SEQ_LEN:
            overflow = len(full_ids) - MAX_SEQ_LEN
            prefix_ids = prefix_ids[overflow:]
            full_ids = prefix_ids + target_ids

        # Build labels: -100 for prefix (masked), actual IDs for target
        labels = [-100] * len(prefix_ids) + target_ids

        # Right-pad to MAX_SEQ_LEN
        pad_len = MAX_SEQ_LEN - len(full_ids)
        attention_mask = [1] * len(full_ids) + [0] * pad_len
        full_ids = full_ids + [tokenizer.pad_token_id] * pad_len
        labels = labels + [-100] * pad_len

        input_ids_list.append(full_ids)
        labels_list.append(labels)
        attention_mask_list.append(attention_mask)

    return Dataset.from_dict({
        "input_ids": input_ids_list,
        "labels": labels_list,
        "attention_mask": attention_mask_list,
    })


dataset = load_and_tokenize(DATA_FILE)
print(f"Dataset: {len(dataset)} examples, padded to {MAX_SEQ_LEN} tokens")

## Verify Loss Masks (Sanity Check)\n\nDecode one example and visually confirm that the prefix/target boundary is correct.\n**Always run this before training.**

In [ ]:
example = dataset[0]
ids = example["input_ids"]
labels = example["labels"]

# Find where labels transition from -100 to real token IDs
mask_boundary = next(i for i, l in enumerate(labels) if l != -100)

# Find where padding starts
pad_start = next((i for i, l in enumerate(labels) if i > mask_boundary and l == -100), len(labels))

prefix_text = tokenizer.decode(ids[:mask_boundary], skip_special_tokens=False)
target_text = tokenizer.decode(ids[mask_boundary:pad_start], skip_special_tokens=False)

print(f"Total tokens: {sum(example['attention_mask'])} (+ {MAX_SEQ_LEN - sum(example['attention_mask'])} padding)")
print(f"Prefix tokens (masked): {mask_boundary}")
print(f"Target tokens (loss): {pad_start - mask_boundary}")
print()
print("=== PREFIX (last 300 chars) ===")
print(prefix_text[-300:])
print()
print("=== TARGET ===")
print(target_text)

## Load Model with 4-bit Quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Resize embeddings for the 8 new special tokens
model.resize_token_embeddings(len(tokenizer))

# Prepare for QLoRA training: freeze base weights, enable gradient checkpointing
model = prepare_model_for_kbit_training(model)

print(f"Model loaded. Parameters: {model.num_parameters():,}")

## Apply LoRA Adapters\n\nTarget the attention projection layers. If `target_modules` causes an error,\nuncomment the diagnostic cell below to discover the correct module names for\nyour model architecture.

In [ ]:
# Diagnostic: uncomment to discover attention module names for your architecture
# for name, _ in model.named_modules():
#     if any(kw in name.lower() for kw in ["attn", "attention", "proj"]):
#         print(name)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Train

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    fp16=False,
    bf16=True,
    logging_steps=1,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

## Save LoRA Adapter + Tokenizer

In [ ]:
adapter_path = OUTPUT_DIR / "final_adapter"
adapter_path.mkdir(parents=True, exist_ok=True)

# Save LoRA adapter weights (small — hundreds of MB, not the full 120B)
model.save_pretrained(str(adapter_path))
# Save tokenizer with the 8 registered special tokens
tokenizer.save_pretrained(str(adapter_path))

print(f"Adapter + tokenizer saved to {adapter_path}")

## Interactive Chat: Simulate an Outbound Sales Call\n\nThe fine-tuned model generates the opening agent line, then you play the customer.\nType `quit` to end the call.

In [ ]:
# System prompt (must match what was used during training)
SYSTEM_PROMPT = (
    "You are an outbound sales agent for American Express.\n"
    "Your goal is to identify customer needs and guide the conversation\n"
    "toward [product/outcome]. Use information provided in the context\n"
    "block when available to reference relevant offers, campaigns,\n"
    "or product details. If no context is provided, rely on the\n"
    "conversation history to guide your response."
)

# Stop token IDs — generation should halt when the model produces these
STOP_TOKEN_IDS = [CUSTOMER_TOKEN_ID, CONV_END_TOKEN_ID, tokenizer.eos_token_id]


def generate_agent_turn(conversation_turns: list[tuple[str, str]]) -> str:
    """Generate the next agent response given conversation history."""

    # Build conversation history string
    conv_lines = []
    for speaker, text in conversation_turns:
        marker = "<|agent|>" if speaker == "agent" else "<|customer|>"
        conv_lines.append(f"{marker}{text}")
    conv_str = "\n".join(conv_lines)

    prompt = (
        f"<|system|>{SYSTEM_PROMPT}<|/system|>\n"
        f"<|context|><|/context|>\n"
        f"<|conversation|>\n"
        f"{conv_str}\n"
        f"<|/conversation|>\n"
        f"<|agent|>"
    )

    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            eos_token_id=STOP_TOKEN_IDS,
        )

    # Decode only the newly generated tokens
    generated_ids = output_ids[0][input_ids.shape[1]:]
    agent_response = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # Truncate at first special token (in case model bleeds into next turn)
    for stop_tok in ["<|customer|>", "<|/conversation|>", "<|agent|>",
                     tokenizer.eos_token]:
        if stop_tok in agent_response:
            agent_response = agent_response[:agent_response.index(stop_tok)]

    return agent_response.strip()

In [ ]:
model.eval()

conversation_turns = []  # list of (speaker, text)

print("=" * 60)
print("OUTBOUND SALES CALL SIMULATION")
print("You are the customer. The agent will speak first.")
print("Type 'quit' to end the conversation.")
print("=" * 60)

while True:
    # Generate agent response
    agent_text = generate_agent_turn(conversation_turns)
    print(f"\nAgent: {agent_text}")
    conversation_turns.append(("agent", agent_text))

    # Get customer (user) input
    user_input = input("\nCustomer: ").strip()
    if user_input.lower() == "quit":
        print("\n[Call ended]")
        break

    conversation_turns.append(("customer", user_input))

---\n\n## (Optional) Load a Saved Adapter for Inference Only\n\nIf restarting the kernel, use this cell to reload the base model + adapter\nwithout retraining.

In [ ]:
# # Uncomment and run this cell to reload from a saved adapter checkpoint
# # (skip the training cells above)
#
# ADAPTER_PATH = "checkpoints/final_adapter"
#
# tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH, local_files_only=True)
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )
#
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_DIR,
#     local_files_only=True,
#     quantization_config=bnb_config,
#     device_map="auto",
#     torch_dtype=torch.bfloat16,
# )
# base_model.resize_token_embeddings(len(tokenizer))
#
# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# model.eval()
#
# # Restore token IDs
# AGENT_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|agent|>")
# CUSTOMER_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|customer|>")
# CONV_END_TOKEN_ID = tokenizer.convert_tokens_to_ids("<|/conversation|>")
# STOP_TOKEN_IDS = [CUSTOMER_TOKEN_ID, CONV_END_TOKEN_ID, tokenizer.eos_token_id]
#
# print("Adapter loaded. Ready for interactive chat.")